&nbsp;

# HN-score Transcript - fanta.bio CRE Intergration (Experimental)

- 全ての転写産物の結果を `join`できるか試してみる

In [2]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

In [3]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261001


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. HN-score Transcript List

In [4]:
HNscore_transcript_all = pl.read_parquet(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HNscore_gencode_human_261001_HN2_transcript_all_gencode_annotation.parquet"
)

display(HNscore_transcript_all)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1
"""ENST00000679223.2""",338,8,60,330,222,1,28,221,"""DNAJB1""","""HGNC:5270""","""protein_coding""","""ENSG00000132002.10""","[""alternative_5_UTR"", ""inferred_transcript_model"", … ""CCDS""]","""-""",5,null,null,"[""polya_site""]","""ensembl_havana_gene_homo_sapie…","[""alternative 5' UTR"", ""inferred transcript model"", ""RNA-Seq supported only""]",3
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…",null,0
"""ENST00000540260.1""",328,9,69,319,216,1,34,215,"""FKBP4""","""HGNC:3720""","""protein_coding_CDS_not_defined""","""ENSG00000004478.9""","[""non_canonical_TEC""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""non canonical TEC""]",1
"""ENST00000676231.2""",324,7,75,317,209,2,40,207,"""HSPB1""","""HGNC:5246""","""protein_coding""","""ENSG00000106211.11""","[""inferred_exon_combination"", ""RNA_Seq_supported_only"", ""basic""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""intropolis_filtered"", ""Snaptron_GTEX"", … ""RNA-Seq supported only""]",4
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000497231.7""",15,266,125,-251,10,182,59,-172,"""ALDH7A1""","""HGNC:877""","""retained_intron""","""ENSG00000164904.19""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapie…",null,0
"""ENST00000490762.5""",12,265,129,-253,10,174,67,-164,"""LUC7L""","""HGNC:6723""","""retained_intron""","""ENSG00000007392.18""",[],"""-""",0,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapie…",null,0
"""ENST00000426094.5""",11,265,130,-254,11,165,75,-154,"""LUC7L""","""HGNC:6723""","""nonsense_mediated_decay""","""ENSG00000007392.18""",[],"""-""",0,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapie…",null,0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. `fanta.bio` resource (BED file)

- directionality, fanta.bioでのアノテーション (PLA,ELA)からの情報を得るためにタブ区切りとして読み込む

In [5]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    n_threads=16,
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
)

# Map exact RGB strings to human-readable names
rgb_to_name = {
    "153,0,0": "dark_red",
    "101,0,0": "dark_red",
    "50,0,0": "dark_red",
    "204,0,0": "red",
    "255,0,0": "red",
    "0,0,51": "dark_blue",
    "0,0,102": "dark_blue",
    "0,0,153": "blue",
    "0,0,204": "blue",
    "0,0,255": "blue",
    "0,0,0": "black",
    "255,202,74": "yellow",
}

fantabio_bed_select = fantabio_bed.with_columns(
    pl.col("itemRgb").replace_strict(
        rgb_to_name,
        default="other"
    ).alias("itemRgb_name")
).select(
    "cre_id",
    "cre_name",
    "directionality",
    "fantabio_cre_class",
    "itemRgb",
    "itemRgb_name"
)

display(fantabio_bed_select)

cre_id,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,str,f64,str,str,str
"""FCHS_1""","""cp1@DDX11L1""",-1.0,"""PLA""","""0,0,255""","""blue"""
"""FCHS_2""","""cp1@WASH7P""",1.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447416""","""cp1@DDX11L17""",-0.684211,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447417""","""FCHS_447417""",0.485714,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_3""","""cp2@LOC101928626""",-1.0,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…
"""FCHS_447311""","""FCHS_447311""",1.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447312""","""FCHS_447312""",0.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447313""","""FCHS_447313""",-1.0,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. `fanta.bio` resource (annotation meta-data)

- GENCODE transcript (ENST からはじまる ID) が近傍 (TSS ± 500b) に存在する シスエレメントのみを選抜する

In [6]:
# Download from https://data.fanta.bio/cre/v1.2.1/human/ in 2026/07/05

fantabio_human_metadata = pl.read_ndjson(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks_annotation.jsonl.gz"
).join(
    fantabio_bed_select,
    on=[
        "cre_id",
        "cre_name"
    ],
    how="inner"
).drop(
    "cre_old_names",
    "tss_ncbi_geneids",
    "tss_hgnc_mgi_ids",
    "tss_uniprot_ids",
    "tss_gene_names",
    "tss_gene_symbols",
    "tss_gene_synonyms",
    "tss_gene_sources"
).with_columns(
    # Q-score は後々フィルタリングの条件に使うかもしれないため、cast(pl.Int64) で整数型に変換しておく
    # print(fanta_bio_human_metadata["chip_atlas_antigens"].dtype) で確認できる
    pl.col("chip_atlas_antigens").list.eval(
        pl.struct([
            pl.element().struct.field("antigen").alias("antigen"),
            pl.element().struct.field("maxqscore").cast(pl.Int64).alias("maxqscore"),
            pl.element().struct.field("experiments").list.eval(
                pl.struct([
                    pl.element().struct.field("id").alias("id"),
                    pl.element().struct.field("qscore").cast(pl.Int64).alias("qscore"),
                ])
            ).alias("experiments"),
        ])
    ).alias("chip_atlas_antigens"),
).explode(
    "tss_transcripts"
).unnest(
    "tss_transcripts"
).filter(
    (pl.col("transcript_id").str.contains("ENST")) # Point1: Ensembl Transcript ID のみに絞る
).with_columns(
    # Point2: "transcript_distance" は str 型で読み込まれるが, int 型に変換しておく
    pl.col("transcript_distance")
    .cast(pl.Float64, strict=False)
    .round(0) # 小数点以下を四捨五入 (8件のみ .5 が存在)
    .cast(pl.Int64)
)

pl.Config.set_tbl_rows(4)
print(f"fanta.bio GENCODE ensembl transcript count: {fantabio_human_metadata.select('transcript_id').n_unique()}")
print(f"fanta.bio GENCODE ensembl transcript-CRE pairs count: {fantabio_human_metadata.select(['transcript_id', 'cre_id']).n_unique()}")
display(fantabio_human_metadata)

fanta.bio GENCODE ensembl transcript count: 411060
fanta.bio GENCODE ensembl transcript-CRE pairs count: 1121578


cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_id,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,str,i64,str,str,list[str],str,str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""FCHS_1""","""chr1""",10610,"""10687""","""cp1@DDX11L1""","[""ENST00000832824.1""]","""434""","""ENST00000832824.1""",434,"[{""EH38E2776517"",""pELS""}]","[{""EGR1"",2552,[{""SRX22828712"",2552}, {""SRX22828708"",1544}, {""SRX22828709"",1187}]}, {""BRD3"",1874,[{""SRX366792"",1874}, {""SRX366793"",621}, {""SRX366794"",537}]}, … {""NRF1"",807,[{""SRX2636389"",807}, {""SRX23881992"",770}, {""SRX1280449"",580}]}]",null,null,null,-1.0,"""PLA""","""0,0,255""","""blue"""
"""FCHS_1""","""chr1""",10610,"""10687""","""cp1@DDX11L1""","[""ENST00000832824.1""]","""434""","""ENST00000832825.1""",438,"[{""EH38E2776517"",""pELS""}]","[{""EGR1"",2552,[{""SRX22828712"",2552}, {""SRX22828708"",1544}, {""SRX22828709"",1187}]}, {""BRD3"",1874,[{""SRX366792"",1874}, {""SRX366793"",621}, {""SRX366794"",537}]}, … {""NRF1"",807,[{""SRX2636389"",807}, {""SRX23881992"",770}, {""SRX1280449"",580}]}]",null,null,null,-1.0,"""PLA""","""0,0,255""","""blue"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_593938""","""chrY""",26409259,"""26409692""","""FCHS_593938""","[""ENST00000725555.1""]","""-6""","""ENST00000344424.4""",123,"[{""EH38E3951258"",""pELS""}, {""EH38E4535230"",""pELS""}]","[{""MYCN"",537,[{""SRX6935376"",537}]}]",null,null,null,0.666667,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_593940""","""chrY""",56881615,"""56881922""","""FCHS_593940""","[""ENST00000810723.1""]","""0""","""ENST00000810723.1""",0,null,"[{""RELA"",1452,[{""SRX5938177"",1452}]}, {""RNF2"",939,[{""SRX18954702"",939}, {""SRX18954704"",699}]}, … {""DLX5"",504,[{""SRX17306262"",504}]}]",null,"[{""hg19::chrY:59027988..59028000,+;hg_201233.1"",""hg_201233.1""}]",null,0.989305,"""PLA""","""204,0,0""","""red"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. `JOIN`



In [7]:
HNscore_transcript_all_fantabio = HNscore_transcript_all.join(
    fantabio_human_metadata,
    on=["transcript_id"],
    how="left", # 対応するCREがない場合は、None となる
    coalesce=True
).sort(
    "HN2",
    descending=True
)

print(f"HNscore_transcript - fantabio CRE pairs count: {HNscore_transcript_all_fantabio.select(['transcript_id', 'cre_id']).n_unique()}")
display(HNscore_transcript_all_fantabio)

HNscore_transcript - fantabio CRE pairs count: 1227198


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapie…",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""
"""ENST00000560179.2""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]","""-""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""RNA-Seq supported partial""]",1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [ ]:
# HNscore_transcript_all_fantabio.write_parquet(
#     f"../../out/fantabio/parquet_file/HNscore_transcript_all_fantabio_v{Execute_date}.parquet",
#     compression_level=9
# )

In [ ]:
# HNscore_transcript_all_fantabio.write_json(
#     f"../../out/fantabio/json_file/HNscore_transcript_all_fantabio_v{Execute_date}.json"
# )